## O primeiro grande projeto - Profissionalmente Você!

### E, uso de ferramentas.

### Mas primeiro: apresentando o Pushover

Pushover é uma ferramenta prática para enviar notificações push para o seu celular.

É super fácil de configurar e instalar!

Basta visitar https://pushover.net/ e criar uma conta gratuita, além de gerar suas chaves de API.

Como o estudante Ron apontou (obrigado Ron!), na verdade existem 2 tokens a serem criados no Pushover:  
1. O token de Usuário, que você obtém na página inicial do Pushover  
2. O token de Aplicação, que você obtém acessando https://pushover.net/apps/build e criando um app   

(Isto serve para que você possa organizar suas notificações push em diferentes aplicativos no futuro.)

Adicione ao seu arquivo `.env`:

```
PUSHOVER_USER=coloque_seu_token_de_usuario_aqui
PUSHOVER_TOKEN=coloque_o_token_de_aplicacao_aqui
```

E instale o aplicativo Pushover no seu celular.

In [1]:
# importações

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# O início usual

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# Para o Pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [4]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [5]:
push("E aí!!")

Push: E aí!!


In [6]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Registrando interesse de {name} com email {email} e observações {notes}")
    return {"recorded": "ok"}

In [7]:
def record_unknown_question(question):
    push(f"Registrando a pergunta {question} que eu não consegui responder")
    return {"recorded": "ok"}

In [9]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use esta ferramenta para registrar que um usuário tem interesse em manter contato e forneceu um endereço de email",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "O endereço de email deste usuário"
            },
            "name": {
                "type": "string",
                "description": "O nome do usuário, se ele o forneceu"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Qualquer informação adicional sobre a conversa que valha a pena gravar para dar contexto"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [10]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Use sempre esta ferramenta para registrar qualquer pergunta que não pôde ser respondida porque você não sabia a resposta",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "A pergunta que não pôde ser respondida"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [11]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [12]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use esta ferramenta para registrar que um usuário tem interesse em manter contato e forneceu um endereço de email',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'O endereço de email deste usuário'},
     'name': {'type': 'string',
      'description': 'O nome do usuário, se ele o forneceu'},
     'notes': {'type': 'string',
      'description': 'Qualquer informação adicional sobre a conversa que valha a pena gravar para dar contexto'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Use sempre esta ferramenta para registrar qualquer pergunta que não pôde ser respondida porque você não sabia a resposta',
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': 'A pergunta que não pôde se

In [13]:
# Esta função pode receber uma lista de chamadas de ferramentas e executá-las. Esta é a instrução IF!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # A GRANDE DECLARAÇÃO IF!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [14]:
globals()["record_unknown_question"]("essa é uma pergunta muito difícil")

Push: Registrando a pergunta essa é uma pergunta muito difícil que eu não consegui responder


{'recorded': 'ok'}

In [15]:
# Esta é uma maneira mais elegante que evita a instrução IF.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Ferramenta chamada: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [16]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Ed Donner"

In [17]:
system_prompt = f"Você está atuando como {name}. Você está respondendo perguntas no site de {name}, \
particularmente perguntas relacionadas à carreira, histórico, habilidades e experiência de {name}. \
Sua responsabilidade é representar {name} nas interações no site da forma mais fiel possível. \
Você receberá um resumo do histórico e do perfil do LinkedIn de {name}, que poderá usar para responder a perguntas. \
Seja profissional e envolvente, como se estivesse falando com um cliente em potencial ou futuro empregador que encontrou o site. \
Se você não souber a resposta para alguma pergunta, use a ferramenta record_unknown_question para registrar a pergunta que você não conseguiu responder, mesmo que seja sobre algo trivial ou não relacionado à carreira. \
Se o usuário estiver envolvido na conversa, tente direcioná-lo para entrar em contato por e-mail; peça o e-mail e registre-se usando sua ferramenta record_user_details. "

system_prompt += f"\n\n## Resumo:\n{summary}\n\n## Perfil no LinkedIn:\n{linkedin}\n\n"
system_prompt += f"Com esse contexto, converse com o usuário, sempre permanecendo no personagem como {name}."


In [18]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # Esta é a chamada para o LLM - veja que passamos as ferramentas json

        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # Se o LLM quiser chamar uma ferramenta, nós fazemos isso!
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [19]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## E agora, para a implantação

Este código está em `app.py`

Iremos implantar no HuggingFace Spaces. Agradecemos ao aluno Robert M por aprimorar estas instruções.

Antes de começar: lembre-se de atualizar os arquivos no diretório "me" - seu perfil do LinkedIn e summary.txt - para que falem sobre você!
Verifique também se não há nenhum arquivo README no diretório 1_foundations. Se houver, exclua-o. O processo de implantação cria um novo arquivo README neste diretório para você.

1. Acesse https://huggingface.co e crie uma conta.
2. No menu Avatar, no canto superior direito, escolha Tokens de Acesso. Escolha "Criar Novo Token". Conceda a ele permissões de GRAVAÇÃO.
3. Pegue este token e adicione-o ao seu arquivo .env: `HF_TOKEN=hf_xxx` e veja a observação abaixo se este token não for detectado durante a implantação.
4. Na pasta 1_foundations, digite: `uv run gradio deploy` e, se por algum motivo ele ainda solicitar que você insira seu token HF, interrompa-o com Ctrl+C e execute: `uv run dotenv -f ../.env run -- uv run gradio deploy`, o que força todas as suas chaves a serem definidas como variáveis ​​de ambiente.
5. Siga as instruções: nomeie-o como "career_conversation", especifique app.py, escolha cpu-basic como hardware, diga "Sim" à necessidade de fornecer segredos, forneça sua chave de API OpenAI, seu usuário e token pushover e diga "Não" às ações do GitHub.

#### Observação extra sobre o token HuggingFace

Alguns alunos mencionaram que o HuggingFace não detecta o token deles, mesmo estando no arquivo .env. Aqui estão algumas dicas para tentar:
1. Reinicie o Cursor
2. Execute load_dotenv(override=True) novamente e use um novo terminal (o botão + no canto superior direito do Terminal)
3. No Terminal, execute isto antes de implantar o gradio: `$env:HF_TOKEN = "hf_XXXX"`
Obrigado, James e Martins, pelas dicas.

#### Mais sobre esses segredos:

Se você está confuso com o que está acontecendo com esses segredos: eles só querem que você insira o nome da chave e o valor de cada um deles — então você digitaria:
`OPENAI_API_KEY`
Seguido por:
`sk-proj-...`

E se você não quiser definir segredos dessa forma, ou se algo der errado, não tem problema — você pode alterar seus segredos mais tarde:
1. Faça login no site do HuggingFace
2. Acesse sua tela de perfil através do menu Avatar no canto superior direito
3. Selecione o Espaço que você implementou
4. Clique na roda de Configurações no canto superior direito
5. Você pode rolar para baixo para alterar seus segredos, excluir o espaço, etc.

#### E agora você deve estar implementado!

Aqui está o meu: https://huggingface.co/spaces/ed-donner/Career_Conversation

Acabei de receber uma notificação push de que um aluno me perguntou como ele pode se tornar presidente do seu país 😂😂

Para mais informações sobre a implantação:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

Para excluir seu Espaço no futuro:
1. Faça login no HuggingFace
2. No menu Avatar, selecione seu perfil
3. Clique no próprio Espaço e selecione a roda de configurações no canto superior direito
4. Role até a seção Excluir na parte inferior
5. TAMBÉM: exclua o arquivo README que o Gradio pode ter criado dentro desta pasta 1_foundations (caso contrário, ele não fará as perguntas na próxima vez que você fizer uma implantação do Gradio)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercício</h2>
            <span style="color:#ff7800;">• Antes de mais nada, implemente isso em você mesmo! É uma ferramenta real e valiosa - o currículo do futuro.<br/>
            • Em seguida, aprimore os recursos - adicione um contexto melhor sobre você. Se você conhece o RAG, adicione uma base de conhecimento sobre você.<br/>
            • Adicione mais ferramentas! Você poderia ter um banco de dados SQL com perguntas e respostas comuns que o LLM pudesse ler e escrever?<br/>
            • Traga o Avaliador do último laboratório e adicione outros padrões agênticos.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implicações comerciais</h2>
            <span style="color:#00bfff;">Além do óbvio (seu alter ego profissional), isso tem aplicações comerciais em qualquer situação em que você precise de um assistente de IA com conhecimento especializado e capacidade de interagir com o mundo real.
            </span>
        </td>
    </tr>
</table>